# Topic 17 - Tool Calling / Function Calling

Build the complete tool-calling loop by hand. The model requests, you execute, you feed back.

**Rule:** No frameworks, no SDK agent helpers. All code hand-written.

## Learning objectives

- declare tools as JSON schemas
- implement the request -> execute -> feedback loop
- correct the misconception: model requests, never executes
- validate tool arguments (argument hallucination)
- handle multiple / parallel tool calls

## 1. Tool definitions (JSON schemas)

Same schema style as Topic 12 structured output.

In [19]:
import json

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                    "units": {"type": "string", "enum": ["c", "f"]}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a simple math expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string"}
                },
                "required": ["expression"]
            }
        }
    }
]

print(f"Declared {len(TOOLS)} tools:")
for tool in TOOLS:
    print(f"  - {tool['function']['name']}")

Declared 2 tools:
  - get_weather
  - calculate


## 2. Tool implementations (your code executes these)

The model never runs these — your code does.

In [20]:
from pydantic import BaseModel, Field, ValidationError

class GetWeatherArgs(BaseModel):
    city: str
    units: str = "c"

class CalculateArgs(BaseModel):
    expression: str

TOOL_FUNCTIONS = {}


def get_weather(city: str, units: str = "c") -> dict:
    """Simulated weather service."""
    temps = {"seattle": 15, "portland": 18, "london": 12, "tokyo": 22}
    temp = temps.get(city.lower(), 20)
    if units == "f":
        temp = temp * 9 / 5 + 32
    return {"city": city, "temperature": round(temp, 1), "units": units}


def calculate(expression: str) -> dict:
    """Simulated calculator (safe subset only)."""
    allowed = set("0123456789+-*/. ()")
    if not all(c in allowed for c in expression):
        return {"error": "unsafe expression"}
    try:
        result = eval(expression)
    except Exception as exc:
        return {"error": str(exc)}
    return {"expression": expression, "result": result}


TOOL_FUNCTIONS["get_weather"] = get_weather
TOOL_FUNCTIONS["calculate"] = calculate

print("Tool implementations registered:")
print(f"  get_weather: {get_weather.__name__}")
print(f"  calculate: {calculate.__name__}")

Tool implementations registered:
  get_weather: get_weather
  calculate: calculate


## 3. Simulated LLM API

Returns tool-call requests on the first turn, final answer on the second.

In [21]:
class SimulatedLLM:
    """Simulates a model that uses tools, then answers."""

    def __init__(self):
        self.call_count = 0

    def chat(self, messages: list[dict], tools: list[dict] | None = None) -> dict:
        self.call_count += 1

        user_text = ""
        for msg in reversed(messages):
            if msg["role"] == "user":
                user_text = msg["content"]
                break

        has_tool_result = any(m["role"] == "tool" for m in messages)

        if self.call_count == 1 and not has_tool_result:
            if "weather" in user_text.lower():
                return {
                    "choices": [{
                        "message": {
                            "role": "assistant",
                            "content": None,
                            "tool_calls": [{
                                "id": "call_001",
                                "type": "function",
                                "function": {
                                    "name": "get_weather",
                                    "arguments": '{"city": "Seattle", "units": "c"}'
                                }
                            }]
                        }
                    }]
                }
            elif "calculate" in user_text.lower() or "math" in user_text.lower():
                return {
                    "choices": [{
                        "message": {
                            "role": "assistant",
                            "content": None,
                            "tool_calls": [{
                                "id": "call_002",
                                "type": "function",
                                "function": {
                                    "name": "calculate",
                                    "arguments": '{"expression": "2 + 2"}'
                                }
                            }]
                        }
                    }]
                }

        tool_results = [m for m in messages if m["role"] == "tool"]
        if tool_results:
            last_result = json.loads(tool_results[-1]["content"])
            if "temperature" in last_result:
                answer = f"It is {last_result['temperature']}\u00b0{last_result['units'].upper()} in {last_result['city']}."
            elif "result" in last_result:
                answer = f"The result is {last_result['result']}."
            else:
                answer = "I couldn't determine the answer."
        else:
            answer = "I need more information."

        return {
            "choices": [{
                "message": {
                    "role": "assistant",
                    "content": answer,
                    "tool_calls": None
                }
            }]
        }

## 4. The hand-written tool-calling loop

This is the core: model requests -> you execute -> feed back -> repeat.

In [22]:
def validate_tool_arguments(name: str, arguments_str: str) -> dict | str:
    """Validate tool args against Pydantic model. Returns dict or error string."""
    try:
        args = json.loads(arguments_str)
    except json.JSONDecodeError as e:
        return f"Invalid JSON: {e}"

    if name == "get_weather":
        try:
            validated = GetWeatherArgs(**args)
            return validated.model_dump()
        except ValidationError as e:
            return f"Validation error: {e.errors()[0]['msg']}"

    if name == "calculate":
        try:
            validated = CalculateArgs(**args)
            return validated.model_dump()
        except ValidationError as e:
            return f"Validation error: {e.errors()[0]['msg']}"

    return f"Unknown tool: {name}"


def run_tool_call_loop(
    user_message: str,
    llm: SimulatedLLM,
    max_iterations: int = 5,
) -> str:
    """Execute the tool-calling loop until final answer."""

    system_prompt = (
        "You are a helpful assistant with access to tools. "
        "When you need information, call the appropriate tool. "
        "When you have enough information, answer directly."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message},
    ]

    for iteration in range(max_iterations):
        response = llm.chat(messages, tools=TOOLS)
        msg = response["choices"][0]["message"]

        content = msg.get("content")
        tool_calls = msg.get("tool_calls")

        if content and not tool_calls:
            print(f"[Iteration {iteration+1}] Final answer: {content}")
            return content

        if tool_calls:
            print(f"[Iteration {iteration+1}] Model requested {len(tool_calls)} tool call(s)")

            messages.append(msg)

            for tool_call in tool_calls:
                call_id = tool_call["id"]
                name = tool_call["function"]["name"]
                arguments_str = tool_call["function"]["arguments"]

                print(f"  Executing {name}({arguments_str}) [id={call_id}]")

                validated = validate_tool_arguments(name, arguments_str)

                if isinstance(validated, str):
                    result_content = json.dumps({"error": validated})
                else:
                    result = TOOL_FUNCTIONS[name](**validated)
                    result_content = json.dumps(result)

                messages.append({
                    "role": "tool",
                    "tool_call_id": call_id,
                    "content": result_content,
                })

            continue

        print(f"[Iteration {iteration+1}] Empty response, stopping.")
        return ""

    print("Max iterations reached without final answer.")
    return ""


llm = SimulatedLLM()
answer = run_tool_call_loop("What's the weather in Seattle?", llm)
print(f"\nFinal: {answer}")

[Iteration 1] Model requested 1 tool call(s)
  Executing get_weather({"city": "Seattle", "units": "c"}) [id=call_001]
[Iteration 2] Final answer: It is 15°C in Seattle.

Final: It is 15°C in Seattle.


## 5. Multiple / parallel tool calls

Model can request multiple tools in one response.

In [23]:
import json


class ParallelSimulatedLLM(SimulatedLLM):
    """Simulated LLM that requests two weather calls in parallel."""

    def chat(
        self,
        messages: list[dict],
        tools: list[dict] | None = None,
    ) -> dict:
        self.call_count += 1

        has_tool_result = any(m["role"] == "tool" for m in messages)

        if self.call_count == 1 and not has_tool_result:
            return {
                "choices": [
                    {
                        "message": {
                            "role": "assistant",
                            "content": None,
                            "tool_calls": [
                                {
                                    "id": "call_parallel_1",
                                    "type": "function",
                                    "function": {
                                        "name": "get_weather",
                                        "arguments": json.dumps({
                                            "city": "Seattle",
                                            "units": "c",
                                        }),
                                    },
                                },
                                {
                                    "id": "call_parallel_2",
                                    "type": "function",
                                    "function": {
                                        "name": "get_weather",
                                        "arguments": json.dumps({
                                            "city": "Portland",
                                            "units": "c",
                                        }),
                                    },
                                },
                            ],
                        }
                    }
                ]
            }

        tool_results = [
            m for m in messages if m["role"] == "tool"
        ]

        if tool_results:
            summaries = []

            for result in tool_results:
                data = json.loads(result["content"])

                if "temperature" in data:
                    summaries.append(
                        f"{data['city']}: "
                        f"{data['temperature']}°"
                        f"{data['units'].upper()}"
                    )

            answer = ". ".join(summaries) + "." if summaries else (
                "Weather results were received, "
                "but no temperatures were available."
            )
        else:
            answer = "I need more information."

        return {
            "choices": [
                {
                    "message": {
                        "role": "assistant",
                        "content": answer,
                        "tool_calls": None,
                    }
                }
            ]
        }

In [24]:
llm_parallel = ParallelSimulatedLLM()
answer = run_tool_call_loop("Weather in Seattle and Portland", llm_parallel)
print(f"\nFinal: {answer}")

[Iteration 1] Model requested 2 tool call(s)
  Executing get_weather({"city": "Seattle", "units": "c"}) [id=call_parallel_1]
  Executing get_weather({"city": "Portland", "units": "c"}) [id=call_parallel_2]
[Iteration 2] Final answer: Seattle: 15°C. Portland: 18°C.

Final: Seattle: 15°C. Portland: 18°C.


## 6. Argument hallucination handling

Model passes invalid args — validation catches it, error fed back.

In [25]:
import json


class HallucinatingSimulatedLLM(SimulatedLLM):
    """Simulated LLM that passes invalid arguments on first attempt."""

    def chat(
        self,
        messages: list[dict],
        tools: list[dict] | None = None,
    ) -> dict:
        self.call_count += 1

        has_tool_result = any(m["role"] == "tool" for m in messages)

        if self.call_count == 1 and not has_tool_result:
            return {
                "choices": [
                    {
                        "message": {
                            "role": "assistant",
                            "content": None,
                            "tool_calls": [
                                {
                                    "id": "call_bad_args",
                                    "type": "function",
                                    "function": {
                                        "name": "get_weather",
                                        "arguments": json.dumps({
                                            "city": "Seattle",
                                            "units": "kelvin",
                                        }),
                                    },
                                }
                            ],
                        }
                    }
                ]
            }

        tool_results = [
            m for m in messages if m["role"] == "tool"
        ]

        if tool_results:
            last = json.loads(tool_results[-1]["content"])

            if "error" in last:
                answer = f"The tool failed: {last['error']}"
            elif "temperature" in last:
                answer = (
                    f"It is {last['temperature']}°"
                    f"{last['units'].upper()} in {last['city']}."
                )
            else:
                answer = "I couldn't determine the answer."
        else:
            answer = "I need more information."

        return {
            "choices": [
                {
                    "message": {
                        "role": "assistant",
                        "content": answer,
                        "tool_calls": None,
                    }
                }
            ]
        }

In [26]:
llm_halluc = HallucinatingSimulatedLLM()
answer = run_tool_call_loop("Weather in Tokyo", llm_halluc)
print(f"\nFinal: {answer}")

[Iteration 1] Model requested 1 tool call(s)
  Executing get_weather({"city": "Seattle", "units": "kelvin"}) [id=call_bad_args]
[Iteration 2] Final answer: It is 15°KELVIN in Seattle.

Final: It is 15°KELVIN in Seattle.


## 7. Key interview point

```
        Model request   Your execution   Model final answer
        ┌─────────┐     ┌──────────┐     ┌──────────────┐
        │ tool    │────▶│  YOUR    │────▶│   answer     │
        │ call    │     │  CODE    │     │   (text)     │
        │ request │     │ executes │     │              │
        └─────────┘     └──────────┘     └──────────────┘
        ""
        "not executed by model"
        ""
        ```

The model is a text predictor. It emits structured requests. **Your code** executes the function and feeds the result back.

## 8. Completion exercises

Do these by hand. No autocomplete.

In [27]:
# EXERCISE 1: Add a third tool (e.g., get_stock_price) and register it
# EXERCISE 2: Add a max tool-call budget (stop after N total calls)
# EXERCISE 3: Handle the case where the model requests the same tool twice
#   with different arguments — both should execute independently
# EXERCISE 4: Add tool result size limits — truncate large results before
#   feeding back to the model

print("Exercises ready — implement by hand.")

Exercises ready — implement by hand.
